# Automobile Price Prediction

predicting car prices using regression models based on specs like engine size, weight etc.

**dataset:** UCI automobile dataset (~205 rows, 26 columns)  
**target:** price  
**models tried:** linear regression, ridge, SVR, PCA-based

## imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.svm import SVR
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score, mean_squared_error
import pickle
import warnings
warnings.filterwarnings('ignore')

## load data

In [ ]:
# dataset doesn't have column names so adding them manually
col_names = [
    'symboling', 'normalized_losses', 'maker', 'fuel_type', 'aspiration',
    'num_of_doors', 'body_style', 'drive_wheels', 'engine_location', 'wheel_base',
    'length', 'width', 'height', 'curb_weight', 'engine_type', 'num_of_cylinders',
    'engine_size', 'fuel_system', 'bore', 'stroke', 'compression_ratio', 'horsepower',
    'peak_rpm', 'city_mpg', 'highway_mpg', 'price'
]

df = pd.read_csv('/content/auto_imports.csv', header=None, names=col_names)
print(df.shape)
df.head()

## data cleaning

In [ ]:
# missing values are stored as '?' instead of NaN
df.replace('?', np.nan, inplace=True)

df.isnull().sum()

In [ ]:
# converting object cols to float before imputing
num_cols = ['normalized_losses', 'bore', 'stroke', 'horsepower', 'peak_rpm']
for col in num_cols:
    df[col] = df[col].astype(float)
    df[col].fillna(df[col].mean(), inplace=True)

# num_of_doors has 2 missing - filling with mode
df['num_of_doors'].fillna('four', inplace=True)

# cant impute target variable, just drop those rows
df.dropna(subset=['price'], inplace=True)
df.reset_index(drop=True, inplace=True)

In [ ]:
# fix dtypes
df[['bore', 'stroke', 'price', 'peak_rpm']] = df[['bore', 'stroke', 'price', 'peak_rpm']].astype(float)
df[['normalized_losses', 'horsepower']] = df[['normalized_losses', 'horsepower']].astype(int)

print('shape after cleaning:', df.shape)
df.dtypes

In [ ]:
df.describe()

## encoding categorical columns

In [ ]:
le = LabelEncoder()
df_enc = df.apply(le.fit_transform)

df_enc.head()

In [ ]:
# just checking what the encoding looks like for a few cols
for col in ['maker', 'fuel_type', 'body_style']:
    print(col)
    print('before:', df[col].unique())
    print('after :', df_enc[col].unique())
    print()

In [ ]:
X = df_enc.drop('price', axis=1)
y = df_enc['price']

df_enc.to_csv('auto_imports_cleaned.csv', index=False)

## feature selection

In [ ]:
selector = SelectKBest(score_func=chi2, k=10)
selector.fit(X, y)

scores_df = pd.DataFrame({'feature': X.columns, 'score': selector.scores_})
scores_df = scores_df.sort_values('score', ascending=False)
print(scores_df.head(12))

In [ ]:
top10 = ['curb_weight', 'length', 'horsepower', 'wheel_base',
         'height', 'engine_size', 'normalized_losses', 'width', 'bore', 'stroke']

X_sel = X[top10].copy()

## EDA

In [ ]:
# checking skewness and kurtosis - want values within reasonable range
for col in top10:
    print(f'{col:25s}  skew: {X_sel[col].skew():.3f}   kurt: {X_sel[col].kurt():.3f}')
print(f"{'price':25s}  skew: {y.skew():.3f}   kurt: {y.kurt():.3f}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 7))
for i, col in enumerate(top10):
    df_enc[col].plot(kind='box', ax=axes[i//5][i%5])
    axes[i//5][i%5].set_title(col)
plt.suptitle('boxplots', y=1.01)
plt.tight_layout()
plt.savefig('boxplots.png', dpi=300)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 7))
for i, col in enumerate(top10):
    sns.histplot(df_enc[col], kde=True, ax=axes[i//5][i%5])
    axes[i//5][i%5].set_title(col)
plt.suptitle('distributions', y=1.01)
plt.tight_layout()
plt.savefig('distributions.png', dpi=300)
plt.show()

In [ ]:
# combining features + target for correlation analysis
plot_df = df_enc[top10 + ['price']].copy()

plt.figure(figsize=(14, 10))
sns.heatmap(plot_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.4)
plt.title('correlation heatmap')
plt.tight_layout()
plt.savefig('heatmap.png', dpi=300)
plt.show()

# curb_weight is correlated with pretty much everything - expected since bigger car = heavier = more expensive

In [ ]:
sns.pairplot(plot_df, plot_kws={'alpha': 0.4, 's': 12})
plt.savefig('pairplot.png', dpi=300)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 7))
for i, col in enumerate(top10):
    sns.residplot(x=df_enc[col], y=df_enc['price'], ax=axes[i//5][i%5],
                  scatter_kws={'s': 8, 'alpha': 0.6})
    axes[i//5][i%5].set_title(col)
plt.suptitle('residual plots', y=1.01)
plt.tight_layout()
plt.savefig('residuals.png', dpi=300)
plt.show()

## modelling

dropping the 3 features with low correlation to price (stroke, normalized_losses, height)
then trying a few different models

In [ ]:
features = ['curb_weight', 'length', 'horsepower', 'wheel_base', 'engine_size', 'width', 'bore']

X_mod = X[features].copy()
X_train, X_test, y_train, y_test = train_test_split(X_mod, y, test_size=0.33, random_state=42)

sc = StandardScaler()
X_train_sc = sc.fit_transform(X_train)
X_test_sc = sc.transform(X_test)

print(X_train.shape, X_test.shape)

### linear regression (baseline)

In [ ]:
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
pred_lr = lr.predict(X_test_sc)

r2_lr = r2_score(y_test, pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, pred_lr))
print(f'R2: {r2_lr:.4f}  RMSE: {rmse_lr:.2f}')

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test, pred_lr, alpha=0.6, s=20)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=1.5)
plt.xlabel('actual'); plt.ylabel('predicted')
plt.title('linear reg - actual vs predicted')
plt.tight_layout()
plt.savefig('lr_pred.png', dpi=300)
plt.show()

### ridge regression

In [ ]:
params = {'alpha': [0.001, 0.1, 1, 10, 100, 1000, 10000, 100000]}
ridge_cv = GridSearchCV(Ridge(), params, scoring='r2', cv=5)
ridge_cv.fit(X_train_sc, y_train)

print('best alpha:', ridge_cv.best_params_['alpha'])

pred_ridge = ridge_cv.best_estimator_.predict(X_test_sc)
r2_ridge = r2_score(y_test, pred_ridge)
rmse_ridge = np.sqrt(mean_squared_error(y_test, pred_ridge))
print(f'R2: {r2_ridge:.4f}  RMSE: {rmse_ridge:.2f}')

### SVR

In [ ]:
svr_params = {
    'C': [0.05, 0.1, 0.15],
    'gamma': [0.5, 1],
    'kernel': ['linear']
}

svr_cv = GridSearchCV(SVR(), svr_params, scoring='r2', cv=5)
svr_cv.fit(X_train_sc, y_train)

print('best params:', svr_cv.best_params_)

pred_svr = svr_cv.best_estimator_.predict(X_test_sc)
r2_svr = r2_score(y_test, pred_svr)
rmse_svr = np.sqrt(mean_squared_error(y_test, pred_svr))
print(f'R2: {r2_svr:.4f}  RMSE: {rmse_svr:.2f}')

### PCA + linear regression

In [ ]:
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print('explained variance:', pca.explained_variance_ratio_.round(3))

X_train_pca_df = pd.DataFrame(X_train_pca, columns=['PC1', 'PC2'])
X_test_pca_df  = pd.DataFrame(X_test_pca,  columns=['PC1', 'PC2'])

plt.figure(figsize=(7, 5))
plt.scatter(X_train_pca_df.PC1, X_train_pca_df.PC2, alpha=0.5, s=15)
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.title('PCA components')
plt.tight_layout(); plt.show()

lr_pca = LinearRegression()
lr_pca.fit(X_train_pca_df, y_train)
pred_pca = lr_pca.predict(X_test_pca_df)

r2_pca = r2_score(y_test, pred_pca)
rmse_pca = np.sqrt(mean_squared_error(y_test, pred_pca))
print(f'R2: {r2_pca:.4f}  RMSE: {rmse_pca:.2f}')

### PCA-guided feature selection + linear regression

using the 5 features that contribute most to PC1 instead of all 7

In [ ]:
pca_features = ['curb_weight', 'length', 'wheel_base', 'horsepower', 'width']
X_pca_sel = X[pca_features].copy()

Xp_train, Xp_test, yp_train, yp_test = train_test_split(X_pca_sel, y, test_size=0.33, random_state=42)

sc2 = StandardScaler()
Xp_train_sc = sc2.fit_transform(Xp_train)
Xp_test_sc  = sc2.transform(Xp_test)

lr2 = LinearRegression()
lr2.fit(Xp_train_sc, yp_train)
pred_combo = lr2.predict(Xp_test_sc)

r2_combo = r2_score(yp_test, pred_combo)
rmse_combo = np.sqrt(mean_squared_error(yp_test, pred_combo))
print(f'R2: {r2_combo:.4f}  RMSE: {rmse_combo:.2f}')

## results

In [ ]:
results = pd.DataFrame({
    'model': ['linear regression', 'ridge', 'SVR', 'PCA + lr', 'PCA combo'],
    'r2': [r2_lr, r2_ridge, r2_svr, r2_pca, r2_combo],
    'rmse': [rmse_lr, rmse_ridge, rmse_svr, rmse_pca, rmse_combo]
})

results['r2_%'] = (results['r2'] * 100).round(2)
results = results.sort_values('r2', ascending=False).reset_index(drop=True)
results

In [ ]:
colors = ['#2ecc71' if i == 0 else '#95a5a6' for i in range(len(results))]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(results['model'], results['r2_%'], color=colors, width=0.5, edgecolor='white')
ax.bar_label(bars, fmt='%.1f%%', padding=3)
ax.set_ylim(results['r2_%'].min() - 3, 100)
ax.set_title('model comparison - R2 score')
ax.set_ylabel('R2 (%)')
plt.xticks(rotation=10)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=300)
plt.show()

In [ ]:
# saving best model
best_idx = results.index[0]
best_name = results.loc[best_idx, 'model']
print('best model:', best_name)

model_lookup = {
    'linear regression': lr,
    'ridge': ridge_cv.best_estimator_,
    'SVR': svr_cv.best_estimator_,
    'PCA + lr': lr_pca,
    'PCA combo': lr2
}

with open('best_model.pkl', 'wb') as f:
    pickle.dump(model_lookup[best_name], f)

## conclusion

PCA combo model gave the best R2 overall. makes sense since curb_weight, length, width, and wheel_base are all basically measuring how big the car is - so using them all together adds noise. picking just the top PCA contributors cleaned that up.

linear models worked better than SVR here which suggests the price relationships are mostly linear.

**things to try next:**
- log transform on price (right skewed)
- try XGBoost / LightGBM
- cross-validation instead of single split